In [1]:
!pip install transformers datasets sentencepiece accelerate torch

import datasets
from datasets import load_dataset, DatasetDict
from transformers import pipeline
import torch
import pandas as pd # 데이터 확인용

# GPU 사용 가능 여부 확인 및 설정 (Colab에서는 보통 GPU 사용 가능)
device = 0 if torch.cuda.is_available() else -1
print(f"사용 가능한 디바이스: {'GPU' if device == 0 else 'CPU'}")

사용 가능한 디바이스: GPU


In [2]:
# imdb 데이터셋 로드
dataset = load_dataset("imdb", split="test")

# 100개 샘플만 선택하여 서브셋 생성
subset = dataset.shuffle().select(range(100))

# 데이터셋 구조 확인
print("선택된 데이터셋 구조:")
print(subset)

# 첫 번째 샘플 확인
print("\n첫 번째 샘플:")
print(subset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

선택된 데이터셋 구조:
Dataset({
    features: ['text', 'label'],
    num_rows: 100
})

첫 번째 샘플:
{'text': "This trio of 30-minute short films on gay-related themes are all quite respectably executed. Each coming-of-age story is played out with pleasant charm and naturalness. This film deserves to be widely distributed and easily obtainable. However, it isn't. I had to order my video copy; none of the local video stores or even the libraries had it in stock.", 'label': 1}


In [3]:
# 1. 요약 파이프라인 (영어 텍스트용)
# 'sshleifer/distilbart-cnn-12-6' 모델을 사용. 영어 텍스트 요약에 특화되어 있음.
summarizer = pipeline(
    task="summarization",
    model="sshleifer/distilbart-cnn-12-6",
    device=device
)

# 2. 감정 분석 파이프라인
# 'SamLowe/roberta-base-go_emotions' 모델 사용
emotion_classifier = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=1,
    device=device
)

# 3. 번역 파이프라인
# 'facebook/nllb-200-distilled-600M' 모델 사용. 영어 -> 한국어 번역에 사용.
translator = pipeline(
    "translation",
    model="facebook/nllb-200-distilled-600M",
    src_lang="eng_Latn", # 원본 언어: 영어
    tgt_lang="kor_Hang", # 대상 언어: 한국어
    device=device
)

print("모든 파이프라인 로드 완료.")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


모든 파이프라인 로드 완료.


In [4]:
# 'subset'은 데이터셋 객체, 'summarizer'는 이미 정의되었다고 가정
def summarize_text(sample):
    # pipeline 호출 시 'truncation=True' 파라미터 추가
    summary = summarizer(
        sample["text"],
        max_length=50,  # 요약 결과의 최대 길이
        min_length=10,  # 요약 결과의 최소 길이
        do_sample=False,
        truncation=True  # 입력 텍스트가 모델의 최대 길이를 초과하면 잘라냅니다.
    )
    sample["summary"] = summary[0]['summary_text']
    return sample

def analyze_emotion(sample):
    # 요약된 영어 텍스트의 감정분석
    result = emotion_classifier(sample["summary"])
    sample["emotion"] = result[0][0]['label']
    return sample

def translate_summary_to_ko(sample):
    # 요약된 영어 텍스트를 한국어로 번역
    result = translator(sample["summary"])
    sample["korean_summary"] = result[0]['translation_text']
    return sample

print("--")
subset = subset.map(summarize_text)
print("--")
subset = subset.map(analyze_emotion)
print("--")
subset = subset.map(translate_summary_to_ko)

print("Data processing complete.")
print(subset.select(range(5)))


--


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


--


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

--


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Data processing complete.
Dataset({
    features: ['text', 'label', 'summary', 'emotion', 'korean_summary'],
    num_rows: 5
})


In [5]:
# 최종 결과를 Pandas DataFrame으로 변환하여 출력
result_df = pd.DataFrame(subset)

# 필요한 컬럼만 선택하여 깔끔하게 출력
final_result = result_df[['text', 'emotion', 'summary', 'korean_summary']]
print(final_result.head())

# 감정 분석 결과의 분포 확인 (선택 사항)
print("\n감정 분석 결과 분포:")
print(final_result['emotion'].value_counts())

                                                text     emotion  \
0  This trio of 30-minute short films on gay-rela...    approval   
1  Engaging characters, nice animation, dynamite ...  admiration   
2  This is a low budget, well acted little gem. A...     neutral   
3  This Blake Edwards film isn't too sure whether...     neutral   
4  I love this film. There is something for every...  admiration   

                                             summary  \
0   This film deserves to be widely distributed a...   
1   There's a lot of excellent humor, but no real...   
2   Alice, a small town Massachusetts teenager, t...   
3   The sheer presence of Julie Andrews, is reaso...   
4   Colin Firth is my favourite actor and so noth...   

                                      korean_summary  
0  이 영화는 널리 보급되고 쉽게 구할 수 있어야 합니다. 저는 비디오 복사본을 주문해...  
1  훌륭한 유머가 있지만 진짜 위협은 없습니다. 그래서 어린이를 걱정하지 마십시오. 예...  
2  마사추세츠의 작은 마을의 십대인 앨리스는 버거를 파는 어머니와 슈퍼마켓에서 체크 아...  
3  줄리 앤드루스의 존재는 이 코미디 드라마 뮤지컬 스파이 

In [6]:
final_dataset = subset

# 최종 결과 확인 (첫 5개 샘플)
for i in range(min(5, len(final_dataset))):
  print(f"\n--- 샘플 {i+1} ---")
  print(f"원문 일부: {final_dataset[i]['text'][:100]}...")
  print(f"요약: {final_dataset[i]['summary']}")
  print(f"한글 번역: {final_dataset[i]['korean_summary']}")
  print(f"감정 분석: {final_dataset[i]['emotion']}")


--- 샘플 1 ---
원문 일부: This trio of 30-minute short films on gay-related themes are all quite respectably executed. Each co...
요약:  This film deserves to be widely distributed and easily obtainable . I had to order my video copy; none of the local video stores or even the libraries had it in stock .
한글 번역: 이 영화는 널리 보급되고 쉽게 구할 수 있어야 합니다. 저는 비디오 복사본을 주문해야 했습니다. 지역 비디오 상점이나 도서관에서도 보관되지 않았습니다.
감정 분석: approval

--- 샘플 2 ---
원문 일부: Engaging characters, nice animation, dynamite songs...all this and cute kitties, too. There's a lot ...
요약:  There's a lot of excellent humor, but no real menace, so don't worry about your little ones . The artwork has a linear quality that may put off some people, but I find it charming . Engaging characters, nice animation
한글 번역: 훌륭한 유머가 있지만 진짜 위협은 없습니다. 그래서 어린이를 걱정하지 마십시오. 예술 작품은 선형적인 품질을 가지고 있습니다.
감정 분석: admiration

--- 샘플 3 ---
원문 일부: This is a low budget, well acted little gem. Alice, a small town Massachusetts teenager, fed up with...
요약:  Alice, a small town

In [13]:
# # CSV 저장
# result_df.to_csv("klue_mrc_processed.csv", index=False, encoding='utf-8-sig')
# print("\n결과를 CSV 파일로 저장했습니다. (필요시 주석 해제)")


결과를 CSV 파일로 저장했습니다. (필요시 주석 해제)
